
# VecDB Maintenance & Indexing
Walk through common lifecycle operations on a VecDB table: stats, pruning, index maintenance, archiving, and cleanup.



## 1. Scenario Overview
- Load connection settings and instantiate a reusable VecDB client.
- Seed demo data, inspect vector metadata, and prune aged entries.
- Request an index rebuild when supported by the SDK build.
- Archive remaining records into a new table before cleaning up.


In [ ]:

%pip install -U oracle-vecdb python-dotenv pandas


In [ ]:

import os
from dotenv import load_dotenv
from oracle_vecdb import OracleVecDB, Configuration

print('Loading VecDB environment variables...')
load_dotenv()

resolved_host = os.getenv('VECDB_REST_URL')
resolved_user = os.getenv('VECDB_USERNAME') or os.getenv('VECDB_USER')
resolved_password = os.getenv('VECDB_PASSWORD')
resolved_access_token = os.getenv('VECDB_ACCESS_TOKEN')
print(f'Resolved REST endpoint: {resolved_host}')
print(f'Resolved user: {resolved_user}')

config_kwargs = {"rest_url": resolved_host}
if resolved_access_token:
    config_kwargs["access_token"] = resolved_access_token
else:
    config_kwargs["username"] = resolved_user
    config_kwargs["password"] = resolved_password
config = Configuration(**config_kwargs)
if os.getenv('VECDB_SELF_SIGNED_SSL', 'false').lower() == 'true':
    config.verify_ssl = False
    import urllib3
    urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)
    print('Disabled SSL verification for self-signed certificates.')

vecdb = OracleVecDB(config)
auth_method = 'bearer token' if config.access_token else 'username/password'
print('Connected to', config.rest_url)
print('Auth method:', auth_method)
print('VecDB client ready for maintenance operations.')



## 2. Create & Seed Maintenance Table
Create a maintenance demo table and populate it with synthetic documents annotated with `AGE_DAYS`.


In [ ]:

from uuid import uuid4
from random import Random

print('Creating maintenance demo tables and seeding sample data...')
MAINT_TABLE = os.getenv('MAINT_TABLE', 'MAINT_DEMO')
ARCHIVE_TABLE = os.getenv('ARCHIVE_TABLE', 'MAINT_ARCHIVE')

vecdb.create_vector_table(
    name=MAINT_TABLE,
    comment='Maintenance demo table',
    annotations={'DOC_ID': 'string', 'AGE_DAYS': 'int'},
)
print(f'Created {MAINT_TABLE}.')

def make_vec(seed):
    rng = Random(seed)
    return [rng.random() for _ in range(12)]

records = [
    {
        'id': str(uuid4()),
        'dense_vector': make_vec(i),
        'metadata': {'DOC_ID': f'D{i}', 'AGE_DAYS': i * 5}
    }
    for i in range(1, 12)
]
vecdb.upsert_vectors(table_name=MAINT_TABLE, vectors=records)
print(f'Inserted {len(records)} demo vectors into {MAINT_TABLE}.')
len(records)



## 3. Describe & Verify Index
Retrieve table statistics to review vector counts and current indexing status before any mutations.


In [ ]:

print('Describing maintenance table to capture vector counts and index status...')
stats = vecdb.describe_vector_table(name=MAINT_TABLE)
stats



## 4. Delete Aged Entries
Remove vectors older than 30 days to simulate compliance or retention rules.


In [ ]:

print('Identifying records older than 30 days for deletion...')
aged_ids = [r['id'] for r in records if r['metadata']['AGE_DAYS'] > 30]
vecdb.delete_vectors(table_name=MAINT_TABLE, ids=aged_ids)
print('Deleted', len(aged_ids), 'rows older than 30 days.')



## 5. Rebuild Index
Kick off an index rebuild using `rebuild_index` to refresh search structures after deletions.


In [ ]:
print('Requesting index rebuild via vecdb.rebuild_index...')
vecdb.rebuild_index(table_name=MAINT_TABLE, index_params={"index_type": "all"})
print('Rebuild requested; monitor VecDB index jobs for completion.')


## 6. Archive Remaining Rows
Copy surviving rows into an archive table so the original can be reset without data loss.


In [ ]:
def query_items(response):
    if isinstance(response, list):
        return response
    if isinstance(response, tuple):
        return list(response)
    return getattr(response, "items", None) or getattr(response, "matches", None) or []


def result_metadata(item):
    return item.get("metadata", {}) if isinstance(item, dict) else getattr(item, "metadata", {})


def result_distance(item):
    if isinstance(item, dict):
        return item.get("distance")
    return getattr(item, "distance", getattr(item, "score", None))


def result_id(item):
    return item.get("id") if isinstance(item, dict) else getattr(item, "id", None)


def result_vector(item):
    if isinstance(item, dict):
        return item.get("vector") or item.get("dense_vector")
    return getattr(item, "vector", getattr(item, "dense_vector", None))


def result_text(item):
    return item.get("text", "") if isinstance(item, dict) else getattr(item, "text", "")



print('Archiving surviving rows into the archive table...')
vecdb.create_vector_table(
    name=ARCHIVE_TABLE,
    comment='Archive for maintenance demo',
    annotations={'DOC_ID': 'string', 'AGE_DAYS': 'int'},
)
remaining = vecdb.query(
    table_name=MAINT_TABLE,
    query_by={'vector': make_vec(1)},
    include_vectors=True,
    top_k=100
)
archive_rows = []
for r in query_items(remaining):
    metadata = result_metadata(r)
    archive_rows.append(
        {
            'id': str(uuid4()),
            'dense_vector': result_vector(r),
            'metadata': {
                'DOC_ID': metadata.get('DOC_ID'),
                'AGE_DAYS': metadata.get('AGE_DAYS'),
            }
        }
    )
vecdb.upsert_vectors(table_name=ARCHIVE_TABLE, vectors=archive_rows)
print(f'Archived {len(archive_rows)} rows into {ARCHIVE_TABLE}.')
len(archive_rows)



## 7. Drop or Reset Tables
Drop both demo tables to leave the VecDB environment clean for the next maintenance run.


In [ ]:

print('Cleaning up maintenance demo tables...')
vecdb.drop_vector_table(name=MAINT_TABLE)
vecdb.drop_vector_table(name=ARCHIVE_TABLE)
print('Cleaned maintenance demo tables.')
